# Gelochip · **Kaizen Architecture**
### RAG (not SFT) for `gf180` RF / mmWave Chip Generation

> **Design intent (this notebook is the spec — the web app follows it).**
> We replace Supervised Fine-Tuning with a self-correcting **Retrieval-Augmented
> Generation** loop. Knowledge lives in **3 local ChromaDB collections**; logic
> lives in a **LangGraph** agent driven by a local **`qwen3.5:9b`** (Ollama).
> Every failed DRC becomes a *lesson* the agent retrieves next time — it gets
> **1 % better every run** (改善 *kaizen*) without ever touching model weights.

---

## 0. Why RAG instead of SFT?

| | SFT (previous approach) | **Kaizen RAG (this design)** |
|---|---|---|
| Knowledge update | re-train (hours, GPU) | `collection.add()` (ms) |
| Risk | catastrophic forgetting of tool-use / Python | none — weights frozen |
| New IP / fix | new dataset + retrain | append one document |
| Hallucinated params | baked into weights | grounded on retrieved templates |
| Cost | high | local + free |

The model stays a **general reasoner**; the *circuit knowledge* is external and
hot-swappable. Corrections are injected as **in-context lessons**, not gradients.

## 1. System architecture

```
┌──────────────────────────────────────────────────────────────────────────┐
│  Gelochip Studio  (web — app/kaizen_app.py + app/static/kaizen/*)          │
│  ┌────────────┐ ┌──────────────┐ ┌────────────┐ ┌──────────────────────┐  │
│  │ Prompt→GDS │ │ IP Library + │ │  Padframe  │ │ Pin-Connect Agent    │  │
│  │   (RF)     │ │ drag-n-drop  │ │ (chipathon)│ │ (wires + pinout names)│  │
│  └─────┬──────┘ └──────┬───────┘ └─────┬──────┘ └──────────┬───────────┘  │
└────────┼───────────────┼───────────────┼──────────────────┼──────────────┘
         │  FastAPI + Server-Sent-Events (live pipeline streaming)           │
┌────────▼───────────────▼───────────────▼──────────────────▼──────────────┐
│  Kaizen LangGraph Agent   (src/gelochip/kaizen/agent.py)                   │
│                                                                            │
│   plan ─▶ retrieve ─▶ generate ─▶ test(build+DRC) ─▶ critic ─┬─(pass)─▶ summarize
│              ▲                                                │           │
│              └──────────────── kaizen_memory ◀──(fail, retries left)──────┘
│   LLM: qwen3.5:9b (Ollama)   ·   embeddings: all-MiniLM-L6-v2 (local)      │
└───────┬───────────────────────────────────────────────────┬──────────────┘
        │ retrieval (parallel)                               │ exec
┌───────▼───────────────────────────────────┐   ┌────────────▼──────────────┐
│  ChromaDB  (notebooks/kaizen_architecture/ │   │  Executor (executor.py)   │
│            chroma_db/)  — 3 collections     │   │  pure-glayout code →      │
│  1. glayout_knowledge                  │   │  gdsfactory Component →   │
│  2. rf_theory                 │   │  GDS → Magic DRC (gf180)→ │
│  3. error_feedback  (kaizen mem.)  │   │  PNG preview              │
└─────────────────────────────────────────────┘   └───────────────────────────┘
```

## 2. The three knowledge collections

One persistent ChromaDB instance, three **named** collections (one DB server,
three tables). Document IDs are human-readable (e.g. `tmpl-current-mirror-00007`,
`theory-bowick-rf-circuit-design-00042`, `lesson-ota-00002`) — never random UUIDs.

| # | Collection name | What it stores | Source datasets |
|---|---|---|---|
| 1 | **`glayout_knowledge`** | parametric, pure-glayout code blocks (instruction → code) | `data/glayout_code/dataset_*.jsonl` |
| 2 | **`rf_theory`** | RF/mmWave books, arXiv abstracts, EE-QA, PySpice corpus | `data/rf_theory/{texts,books,arxiv,huggingface,analog_pyspice}` |
| 3 | **`error_feedback`** | Kaizen memory: scenario → error → root cause → **fix** | `lessons_seed.json` + grown live by the agent |

> The on-disk `chroma_db/<uuid>/` folders are ChromaDB's internal *segment*
> storage — that UUID is unavoidable and **is not** the collection name or doc ID.

In [1]:
import os, sys
os.environ.setdefault("PDK_ROOT", os.path.expanduser("~/pdks"))
sys.path.insert(0, os.path.abspath("../../src"))   # repo_root/src

from gelochip.kaizen import config, collections
print("ChromaDB dir :", config.CHROMA_DIR)
print("Embed model  :", config.EMBED_MODEL)
print("LLM (agent)  :", config.LLM_MODEL, "via", config.OLLAMA_BASE_URL)
print("Collections  :", config.ALL_COLLECTIONS)

ChromaDB dir : /home/irman/Gelochip/notebooks/kaizen_architecture/chroma_db
Embed model  : sentence-transformers/all-MiniLM-L6-v2
LLM (agent)  : qwen3.5:9b via http://localhost:11434
Collections  : ('glayout_knowledge', 'rf_theory', 'error_feedback')


In [2]:
# Build the 3 collections from the datasets (idempotent — re-run to refresh).
# First run downloads the all-MiniLM-L6-v2 embedding model (~80 MB).
if max(collections.collection_counts().values()) == 0:
    print("Ingesting…", collections.build_all())
print("Live chunk counts:", collections.collection_counts())

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Live chunk counts: {'glayout_knowledge': 216, 'rf_theory': 6231, 'error_feedback': 0}


In [3]:
# Inspect what is actually inside each collection (readable IDs + metadata).
for name in config.ALL_COLLECTIONS:
    vs = collections.get_vectorstore(name)
    got = vs._collection.get(limit=2)
    print(f"\n=== {name}  (total {vs._collection.count()}) ===")
    for did, meta in zip(got["ids"], got["metadatas"]):
        print(f"  {did}  ->  {meta}")


=== glayout_knowledge  (total 216) ===
  tmpl-current-mirror-00000  ->  {'doc_type': 'code_template', 'source': 'dataset_circuits.jsonl', 'circuit': 'current_mirror', 'pdk': 'gf180'}
  tmpl-current-mirror-00001  ->  {'source': 'dataset_circuits.jsonl', 'doc_type': 'code_template', 'circuit': 'current_mirror', 'pdk': 'gf180'}

=== rf_theory  (total 6231) ===
  theory-bowick-rf-circuit-design-00000  ->  {'source': 'bowick_rf_circuit_design_2e', 'doc_type': 'book'}
  theory-bowick-rf-circuit-design-00001  ->  {'doc_type': 'book', 'source': 'bowick_rf_circuit_design_2e'}

=== error_feedback  (total 0) ===


## 3. Retrieval (the RAG core)

A single local sentence-transformer (`all-MiniLM-L6-v2`, 384-dim, normalized)
embeds both the datasets and the live query — **no API keys, fully offline**.
The agent queries all three collections and concatenates the hits into the
generation prompt.

In [4]:
q = "NMOS current mirror with ratio 2 on gf180, DRC clean"
hit = collections.get_vectorstore(config.COLL_TEMPLATES).similarity_search(q, k=1)[0]
print("Top glayout_knowledge hit:", hit.metadata)
print(hit.page_content[:400], "…")

# error_feedback starts EMPTY — it grows as the Kaizen loop logs error→fix lessons.
fb = collections.get_vectorstore(config.COLL_LESSONS).similarity_search(q, k=1)
print("\nTop error_feedback hit:",
      fb[0].page_content if fb else "(collection empty — populated at runtime)")

Top glayout_knowledge hit: {'pdk': 'gf180', 'status': 'verified_correct', 'circuit': 'ota', 'source': 'ota_clean.py', 'doc_type': 'clean_circuit'}
# Task
Generate DRC-clean glayout code for a ota on gf180 PDK.

# glayout solution (DRC-verified)
from glayout.pdk.gf180_mapped import gf180_mapped_pdk as PDK
from glayout.primitives.fet import nmos, pmos
from glayout.routing.c_route import c_route
from glayout.routing.straight_route import straight_route
from glayout.util.comp_utils import evaluate_bbox
from gdsfactory.component import Component
 …



Top error_feedback hit: (collection empty — populated at runtime)


## 4. The Kaizen agent (LangGraph)

A 6-node state machine. The **critic** grades the layout against the *real* DRC
result; on failure it routes through **`kaizen_memory`**, which distils a
`scenario → error → fix` lesson and writes it back into
`error_feedback` so the *next* `generate` retrieves the correction —
in-context learning instead of SFT.

| Node | Function | Model / tool |
|---|---|---|
| `plan` | classify circuit + 3–5 step layout plan | `qwen3.5:9b` |
| `retrieve` | parallel top-k over the 3 collections | MiniLM + Chroma |
| `generate` | emit **pure-glayout** code grounded on templates | `qwen3.5:9b` (RAG) |
| `test` | exec code → GDS → **gf180 Magic DRC** → PNG | `executor.py` |
| `critic` | pass / fail routing (DRC-gated) | — |
| `kaizen_memory` | write `error→fix` lesson, loop back | `qwen3.5:9b` + Chroma |
| `summarize` | final verdict + artifact paths | `qwen3.5:9b` |

In [5]:
from gelochip.kaizen import agent
g = agent.build_graph()
try:
    print(g.get_graph().draw_ascii())
except Exception as e:
    print("(ascii draw needs grandalf)", e)
    print("nodes:", list(g.get_graph().nodes))

            +-----------+              
            | __start__ |              
            +-----------+              
                  *                    
                  *                    
                  *                    
              +------+                 
              | plan |                 
              +------+                 
                  *                    
                  *                    
                  *                    
            +----------+               
            | retrieve |               
            +----------+               
                  *                    
                  *                    
                  *                    
            +----------+               
            | generate |               
            +----------+               
           ***         ***             
          *               *            
        **                 ***         
  +------+                    *        


## 5. Executor — generate → build GDS → DRC → preview

The agent's **test** step. It executes the generated **pure-glayout** code
(exactly the style of the SFT datasets — `from glayout… import`,
`gf180_mapped_pdk`), finds the resulting `gdsfactory.Component`, writes a GDS,
runs **Magic DRC** for gf180, and renders a PNG. Trailing ngspice harness code is
tolerated (the layout is recovered even if a later line raises). DRC errors are
**non-fatal** — they are reported and fed back to the next generation, or left
for manual correction.

In [6]:
# Executor smoke test on a tiny pure-gdsfactory component (fast, no LLM, no glayout build).
from gelochip.kaizen import executor
demo = '''
import gdsfactory as gf
component = gf.Component("demo_block")
component.add_polygon([(0,0),(20,0),(20,8),(0,8)], layer=(1,0))
'''
res = executor.run_layout_code(demo, str(config.OUTPUT_DIR/"nb_demo"), name="demo", run_drc=False)
print({k: res[k] for k in ("stage","ok","passed","gds_path","png_path")})

2026-05-28 12:58:05.040 | WARNING  | gdsfactory.pdk:get_active_pdk:733 - No active PDK. Activating generic PDK.



2026-05-28 12:58:05.101 | INFO     | gdsfactory.technology.layer_views:__init__:790 - Importing LayerViews from YAML file: '/home/irman/Gelochip/.venv/lib/python3.13/site-packages/gdsfactory/generic_tech/layer_views.yaml'.


2026-05-28 12:58:05.102 | INFO     | gdsfactory.pdk:activate:337 - 'generic' PDK is now active


2026-05-28 12:58:05.103 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/home/irman/Gelochip/outputs/kaizen/nb_demo/demo.gds'


{'stage': 'done', 'ok': True, 'passed': True, 'gds_path': '/home/irman/Gelochip/outputs/kaizen/nb_demo/demo.gds', 'png_path': '/home/irman/Gelochip/outputs/kaizen/nb_demo/demo.png'}


## 6. Run the full Kaizen loop

Requires Ollama serving `qwen3.5:9b`. This calls the LLM several times and runs a
real glayout build + DRC, so it can take a few minutes. Set `RUN_AGENT = True`.

In [7]:
RUN_AGENT = False   # ← flip to True to run end-to-end (needs Ollama + time)

if RUN_AGENT:
    def on_event(ev):  # live trace
        print(f"[{ev['node']:>14}] {ev['msg'][:90]}")
    state = agent.run("Design a gf180 NMOS current mirror with ratio 2", on_event=on_event)
    print("\n=== FINAL ===\n", state["answer"])
else:
    print("RUN_AGENT is False — flip it to execute the full plan→retrieve→generate→DRC→kaizen loop.")

RUN_AGENT is False — flip it to execute the full plan→retrieve→generate→DRC→kaizen loop.


## 7. Gelochip Studio — the web platform (4 tools)

The web UI follows this architecture exactly. One FastAPI backend
(`app/kaizen_app.py`) + a vanilla HTML/CSS/JS frontend
(`app/static/kaizen/`), streaming every pipeline event over SSE.

### Tool 1 — Prompt → GDSII (RF)
Type a spec, watch `plan → retrieve → generate → DRC → kaizen` stream live, see
the **GDS preview**, DRC verdict, and the generated glayout code. Download GDS.

### Tool 2 — IP Library + drag-n-drop chip canvas
A **database of built IPs** (`src/gelochip/kaizen/ip_library.py`) scanned from
`data/circuits/*` (current_mirror, diff_pair, ota, opamp, n_block, p_block,
fvf, transmission_gate, …) plus any agent-generated block. Each IP card carries
its **bbox + pins**. Drag cards onto the chip floorplan to place them.

### Tool 3 — Padframe
A `gf180` pad ring as the chip boundary. We integrate the **SSCS Chipathon 2025**
"Blocks & Bots" collateral and the **caravel-gf180mcu** padframe; a generated
fallback pad ring is provided so the canvas works offline.
Sources: `github.com/sscs-ose/sscs-chipathon-2025`, `github.com/efabless/caravel-gf180mcu`.

### Tool 4 — Pin-Connect agentic AI
Given the placed blocks and their pins, an agent proposes the **net list**
(which pin connects to which), assigns **pinout names**, and draws the **wires**
on the canvas — then exports a connectivity/netlist artifact.

### API surface

| Endpoint | Purpose |
|---|---|
| `POST /api/kaizen/run` + `GET /api/kaizen/stream/{job}` | Tool 1: run agent, stream events |
| `GET /api/kaizen/collections` | live collection counts |
| `GET /api/ip/library` | Tool 2: list IP blocks (bbox + pins) |
| `GET /api/padframe` | Tool 3: padframe pads + outline |
| `POST /api/connect` | Tool 4: agent proposes pin↔pin wiring + names |

## 8. Data provenance

**Collection 1 — code templates** ← `data/glayout_code/`
`dataset_circuits.jsonl`, `dataset_full.jsonl`, `dataset_operations.jsonl`,
`dataset_circuits_drc_only.jsonl`, `dataset_full_drc_only.jsonl`
(human → pure-glayout code pairs, gf180).

**Collection 2 — RF/mmWave theory** ← `data/rf_theory/`
- `texts/` — Steer (v0–v5), Bowick *RF Circuit Design 2e*, MIT 6.013, NIST RF power
- `books/*.pdf` — same titles (parse with `--pdfs`)
- `arxiv/all_abstracts.txt` + `arxiv/pdfs/*.pdf`
- `huggingface/*.jsonl` — STEM-AI EE, MMLU electrical engineering
- `analog_pyspice/sft_pairs.jsonl` — AnalogCoder PySpice examples

**Collection 3 — lessons** ← `lessons_seed.json` (known netlist/DRC/PDK bugs) +
runtime kaizen writes.

## 9. How to run

```bash
# one-time: build the 3 collections
.venv/bin/python scripts/kaizen_ingest.py          # add --pdfs to parse raw PDFs

# make sure Ollama serves the agent model
ollama pull qwen3.5:9b

# launch Gelochip Studio
.venv/bin/python -m uvicorn app.kaizen_app:app --port 8090 --reload
#  → http://localhost:8090/
```

## 10. Kaizen roadmap

1. **Log every failure** → `error_feedback` (done, automatic).
2. **Gold set** of 10–20 must-pass prompts; re-run on every change.
3. Metadata tagging (`status: verified_correct | historical_error`) so the agent
   filters bad memories — already on each lesson.
4. Promote DRC-clean generated blocks into the **IP Library** automatically.